In [1]:
# Shared project setup for imports and file locations
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / 'data'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
FIGURES_DIR = PROJECT_ROOT / 'figures'

def resolve_path(path):
    candidate = Path(path)
    if candidate.exists():
        return candidate
    text = str(path).replace('\\', '/')
    name = Path(text).name
    special = {
        'positive_controls.pkl': ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl',
        'negative_controls.pkl': ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl',
        'Ten_positive_controls_1119.pkl': ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl',
        'Ten_negative_controls_1119.pkl': ARTIFACTS_DIR / 'controls' / 'negative_controls.pkl',
        'fcg.txt': DATA_DIR / 'fcg.txt',
    }
    if name in special:
        return special[name]
    matches = [p for p in PROJECT_ROOT.rglob(name) if '.ipynb_checkpoints' not in p.parts and '.git' not in p.parts]
    if len(matches) == 1:
        return matches[0]
    if (text.startswith('/Users/') or text.startswith('/home/') or ':\\' in text) and '.' not in name:
        return PROJECT_ROOT
    return candidate

from pdm_learn.preprocessing import build_density_map, density_centers, densitymap, drop_nan, extract, mut_trim, normalize, trim, trim_pairs
from pdm_learn.modeling import KFold_PR, LOOCV, LOOCV_grouped_plot, area_table, core_predict, heatmap, importance_test, ks_pvalue
from pdm_learn.simulation import eps, partition


In [2]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
from bisect import bisect_left
import pickle


In [3]:
gene_exp = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'Gene_Expression_Trimmed.csv')
gene_exp.name = 'gene_exp'

copy_num = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'Copy_Number_Trimmed.csv')
copy_num.name = 'copy_num'

shRNA = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'shRNA_Trimmed.csv')
shRNA.name = 'shRNA'

gene_mut = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'Gene_Mutation_Trimmed.csv')
gene_mut.name = 'gene_mut'

CRISPR = pd.read_csv(DATA_DIR / 'DepMap_Trimmed' / 'CRISPR_Trimmed.csv')
CRISPR.name = 'CRISPR'


In [4]:
biogrid_pairs = (
    pd.read_csv(DATA_DIR / 'clean_biogrid_interactions.csv')
    .rename(columns={'Gene_A': 'Gene1', 'Gene_B': 'Gene2'})
)
biogrid_pairs['Gene1'] = biogrid_pairs['Gene1'].astype(str).str.strip()
biogrid_pairs['Gene2'] = biogrid_pairs['Gene2'].astype(str).str.strip()
biogrid_pairs = biogrid_pairs.dropna(subset=['Gene1', 'Gene2']).reset_index(drop=True)


In [5]:
genes = sorted(set(gene_exp.iloc[:,0].str.strip()) & 
               set(copy_num.iloc[:,0].str.strip()) & 
               set(shRNA.iloc[:,0].str.strip()) & 
               set(CRISPR.iloc[:,0].str.strip()))
genes = [i.strip() for i in genes]


In [6]:
datasets = gene_exp, copy_num, shRNA, gene_mut, CRISPR
points = density_centers(gene_exp, 7), [0,1,2,3,4,6,8], density_centers(shRNA, 7), [0, 1], density_centers(CRISPR, 7)
cont = True, False, True, False, True


In [7]:
a = ['CCND1.CCNE1', 'CCNA2.CCND1', 'CCNB1.CCND1', 'CCNA2.CCNE1', 'CCNB1.CCNE1', 'CDK1.CDK4', 'CDK1.CDK6', 'CDK2.CDK4', 'CCNB1.CDK4', 'CCNB1.CDK6', 'RB1.RBL1', 'E2F2.E2F3', 'E2F2.HDAC1', 'E2F2.RBL1', 'E2F3.HDAC1', 'MCM3.ORC1', 'MCM4.ORC1', 'MCM5.ORC1', 'MCM6.ORC1', 'CDT1.ORC1', 'MCM4.ORC2', 'MCM6.ORC2', 'MCM3.ORC3', 'MCM4.ORC3', 'MCM5.ORC3', 'MCM6.ORC3', 'CDT1.ORC3', 'MCM2.ORC4', 'MCM4.ORC4', 'MCM5.ORC4', 'MCM6.ORC4', 'MCM7.ORC4', 'CDT1.ORC4', 'MCM4.ORC5', 'MCM5.ORC5', 'MCM6.ORC5', 'MCM7.ORC5', 'CDT1.ORC5', 'MCM3.ORC6', 'MCM5.ORC6', 'MCM6.ORC6', 'MCM7.ORC6', 'CDT1.ORC6', 'CDT1.MCM3', 'CDC6.MCM4', 'CDT1.MCM5', 'CDC6.MCM6', 'CDT1.MCM7', 'ANAPC1.ANAPC3', 'ANAPC1.ANAPC6', 'ANAPC1.ANAPC8', 'ANAPC1.ANAPC9', 'ANAPC1.ANAPC11', 'ANAPC1.CDH1', 'ANAPC2.ANAPC3', 'ANAPC2.ANAPC6', 'ANAPC2.ANAPC8', 'ANAPC2.ANAPC9', 'ANAPC2.CDH1', 'ANAPC3.ANAPC4', 'ANAPC3.ANAPC5', 'ANAPC3.ANAPC6', 'ANAPC3.ANAPC7', 'ANAPC3.ANAPC8', 'ANAPC3.ANAPC9', 'ANAPC10.ANAPC3', 'ANAPC11.ANAPC3', 'ANAPC3.CDC20', 'ANAPC3.CDH1', 'ANAPC4.ANAPC6', 'ANAPC4.ANAPC8', 'ANAPC4.ANAPC9', 'ANAPC4.CDH1', 'ANAPC5.ANAPC6', 'ANAPC5.ANAPC8', 'ANAPC5.ANAPC9', 'ANAPC11.ANAPC5', 'ANAPC5.CDH1', 'ANAPC6.ANAPC7', 'ANAPC6.ANAPC8', 'ANAPC6.ANAPC9', 'ANAPC10.ANAPC6', 'ANAPC11.ANAPC6', 'ANAPC6.CDC20', 'ANAPC6.CDH1', 'ANAPC7.ANAPC8', 'ANAPC7.ANAPC9', 'ANAPC11.ANAPC7', 'ANAPC7.CDH1', 'ANAPC8.ANAPC9', 'ANAPC10.ANAPC8', 'ANAPC11.ANAPC8', 'ANAPC8.CDC20', 'ANAPC8.CDH1', 'ANAPC10.ANAPC9', 'ANAPC11.ANAPC9', 'ANAPC9.CDC20', 'ANAPC9.CDH1', 'ANAPC10.CDH1', 'ANAPC11.CDH1', 'MRE11.RAD50', 'MRE11.NBN', 'BARD1.PALB2', 'FANCB.FANCE', 'FANCB.FANCD2', 'FANCE.FANCL', 'FANCD2.FANCF', 'ATR.ATRIP', 'ATRIP.CHEK1', 'ATRIP.TOPBP1', 'ATRIP.CLSPN', 'CHEK1.TOPBP1', 'CLSPN.TOPBP1', 'PIK3CA.PIK3CB', 'AKT2.PIK3CA', 'MTOR.PIK3CA', 'PIK3CA.RICTOR', 'PIK3CA.RPTOR', 'MLST8.PIK3CA', 'PIK3CA.TSC1', 'PIK3CA.TSC2', 'AKT1.PIK3CB', 'AKT2.PIK3CB', 'MTOR.PIK3CB', 'PIK3CB.RICTOR', 'PIK3CB.RPTOR', 'MLST8.PIK3CB', 'PIK3CB.TSC1', 'PIK3CB.TSC2', 'AKT1.AKT2', 'AKT1.RPTOR', 'AKT1.MLST8', 'AKT1.TSC2', 'AKT2.MTOR', 'AKT2.RICTOR', 'AKT2.RPTOR', 'AKT2.MLST8', 'AKT2.TSC1', 'AKT2.TSC2', 'MTOR.TSC1', 'MTOR.TSC2', 'RICTOR.TSC1', 'RICTOR.TSC2', 'RPTOR.TSC1', 'RPTOR.TSC2', 'MLST8.TSC1', 'MLST8.TSC2', 'KRAS.MAPK1', 'MAP2K1.NRAS', 'MAPK1.NRAS', 'PRKAA1.PRKAA2', 'BCL2.MCL1', 'BCL2L1.MCL1', 'BAX.BCL2L11', 'BAK1.BID', 'BAK1.BCL2L11', 'BCL2L11.BID', 'CASP9.CYCS', 'IL1B.NLRP3', 'IL1B.PYCARD', 'ATG13.ULK1', 'PIK3C3.ULK1', 'ATG5.ULK1', 'ATG7.ULK1', 'ATG13.RB1CC1', 'ATG13.BECN1', 'ATG13.PIK3C3', 'ATG13.ATG5', 'ATG13.ATG7', 'BECN1.RB1CC1', 'PIK3C3.RB1CC1', 'ATG7.RB1CC1', 'ATG5.BECN1', 'ATG7.BECN1', 'ATG5.PIK3C3', 'ATG7.PIK3C3', 'ARID1A.BRD9', 'ARID1B.PBRM1', 'ARID1B.BRD9', 'BRD9.SMARCB1', 'BRD9.PBRM1', 'EPC1.EPC2', 'ATM.MDM4', 'CHEK2.MDM4', 'MDM4.SFN', 'ATM.SFN', 'CHEK2.SFN', 'PSMA1.PSMD9', 'PSMA1.PSMD10', 'PSMA2.PSMD9', 'PSMA2.PSMD10', 'PSMA3.PSMD9', 'PSMA3.PSMD10', 'PSMA4.PSMD9', 'PSMA4.PSMD10', 'PSMA5.PSMD9', 'PSMA5.PSMD10', 'PSMA6.PSMD9', 'PSMA6.PSMD10', 'PSMA7.PSMD9', 'PSMA7.PSMD10', 'PSMB1.PSMD9', 'PSMB1.PSMD10', 'PSMB2.PSMD9', 'PSMB2.PSMD10', 'PSMB3.PSMD9', 'PSMB3.PSMD10', 'PSMB4.PSMD10', 'PSMB5.PSMD9', 'PSMB5.PSMD10', 'PSMB6.PSMD9', 'PSMB6.PSMD10', 'PSMB7.PSMD9', 'PSMB7.PSMD10', 'PSMC1.PSMD9', 'HSP90AA1.STIP1', 'HSP90AB1.STIP1', 'CDC37.STIP1', 'ATF4.EIF2S1', 'EIF2AK4.EIF2S1', 'DDIT3.EIF2S1', 'ATF4.EIF2AK4', 'DDIT3.EIF2AK4']
x = []
y = []
for pair in a:
    g1, g2 = pair.split('.')
    x.append(g1)
    y.append(g2)

biogrid_pairs = pd.DataFrame({'Gene1' : x, 'Gene2' : y})
biogrid_pairs

,Gene1,Gene2
0,CCND1,CCNE1
1,CCNA2,CCND1
2,CCNB1,CCND1
3,CCNA2,CCNE1
4,CCNB1,CCNE1
...,...,...
218,ATF4,EIF2S1
219,EIF2AK4,EIF2S1
220,DDIT3,EIF2S1
221,ATF4,EIF2AK4


In [8]:
temp_time = time.time()
pair_mask = biogrid_pairs['Gene1'].isin(genes) & biogrid_pairs['Gene2'].isin(genes)
pair_table = biogrid_pairs.loc[pair_mask].reset_index(drop=True)
pair_matrix = build_density_map(
    datasets,
    pair_table[['Gene1', 'Gene2']].to_numpy().tolist(),
    points,
    cont,
)
pair_matrix = pd.concat([pair_table, pair_matrix], axis=1)
print(f"converted {len(pair_table)} pairs in {time.time() - temp_time:.2f}s")


converted 166 pairs in 125.33s


In [9]:
output_path = ARTIFACTS_DIR / 'results' / 'clean_biogrid_interactions_pdm_missing.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
pair_matrix.to_csv(output_path, index=False)
output_path


PosixPath('/Users/jzx/Documents/Computer Science/PDM_Learn/artifacts/results/clean_biogrid_interactions_pdm_missing.csv')

In [10]:
df = pd.concat([pd.read_csv(ARTIFACTS_DIR / 'results' / 'clean_biogrid_interactions_pdm.csv'), 
                pd.read_csv(ARTIFACTS_DIR / 'results' / 'clean_biogrid_interactions_pdm_missing.csv')], axis=0, ignore_index=True)
df

,Gene1,Gene2,Interaction_Type,Confidence_Score,pair,gene_exp.gene_exp.0,gene_exp.gene_exp.1,gene_exp.gene_exp.2,gene_exp.gene_exp.3,gene_exp.gene_exp.4,...,CRISPR.CRISPR.39,CRISPR.CRISPR.40,CRISPR.CRISPR.41,CRISPR.CRISPR.42,CRISPR.CRISPR.43,CRISPR.CRISPR.44,CRISPR.CRISPR.45,CRISPR.CRISPR.46,CRISPR.CRISPR.47,CRISPR.CRISPR.48
0,MAP2K4,FLNC,direct interaction,NaN,MAP2K4.FLNC,-7.174697,-7.173454,-7.154708,-7.105875,-7.161410,...,-5.686287,-6.483615,-6.670788,-6.672022,-6.668549,-6.636919,-6.526064,-6.306870,-6.570739,-6.671530
1,GATA2,PML,direct interaction,NaN,GATA2.PML,-7.174724,-7.174582,-7.173217,-7.172881,-7.173199,...,-5.656049,-6.546896,-6.671619,-6.672032,-6.671773,-6.664401,-6.642069,-6.623849,-6.666457,-6.672020
2,RPA2,STAT3,direct interaction,NaN,RPA2.STAT3,-7.174701,-7.170263,-7.113910,-6.886640,-7.118832,...,-5.235949,-6.266843,-6.659840,-6.671994,-6.661821,-6.314082,-6.310918,-6.555979,-6.650990,-6.671916
3,ARF1,GGA3,direct interaction,NaN,ARF1.GGA3,-7.174668,-7.170583,-7.166090,-7.169137,-7.174248,...,-5.748181,-6.438846,-6.645946,-6.671992,-6.667723,-6.383032,-6.171960,-6.645068,-6.671532,-6.671986
4,ARF3,ARFIP2,direct interaction,NaN,ARF3.ARFIP2,-7.174679,-7.171916,-7.138842,-6.930789,-7.133416,...,-5.497347,-6.187912,-6.661641,-6.672033,-6.671995,-6.670678,-6.642824,-6.455816,-6.616414,-6.671438
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85103,ATF4,EIF2S1,NaN,NaN,ATF4.EIF2S1,-7.174693,-7.151553,-6.887392,-7.046526,-7.143155,...,-4.713647,-6.167011,-6.666817,-6.671111,-6.638868,-6.498736,-6.389537,-6.493916,-6.645644,-6.671898
85104,EIF2AK4,EIF2S1,NaN,NaN,EIF2AK4.EIF2S1,-7.173398,-7.130653,-7.007865,-6.944225,-7.137144,...,-4.385949,-5.853997,-6.641572,-6.671355,-6.646867,-6.577753,-6.407869,-6.397464,-6.643981,-6.671680
85105,DDIT3,EIF2S1,NaN,NaN,DDIT3.EIF2S1,-7.174652,-7.149601,-6.897219,-7.036838,-7.143299,...,-5.092874,-6.566535,-6.671867,-6.672004,-6.663907,-6.526064,-6.324313,-6.501185,-6.667402,-6.672030
85106,ATF4,EIF2AK4,NaN,NaN,ATF4.EIF2AK4,-7.174724,-7.173997,-7.145300,-7.085000,-7.156027,...,-4.262400,-5.579935,-6.638003,-6.672006,-6.669443,-6.629798,-6.266319,-6.267882,-6.624130,-6.671541


In [11]:
df.to_csv(ARTIFACTS_DIR / 'results' / 'clean_biogrid_interactions_pdm_combined.csv', index=False)

In [12]:
pair_matrix.head()


,Gene1,Gene2,pair,gene_exp.gene_exp.0,gene_exp.gene_exp.1,gene_exp.gene_exp.2,gene_exp.gene_exp.3,gene_exp.gene_exp.4,gene_exp.gene_exp.5,gene_exp.gene_exp.6,...,CRISPR.CRISPR.39,CRISPR.CRISPR.40,CRISPR.CRISPR.41,CRISPR.CRISPR.42,CRISPR.CRISPR.43,CRISPR.CRISPR.44,CRISPR.CRISPR.45,CRISPR.CRISPR.46,CRISPR.CRISPR.47,CRISPR.CRISPR.48
0,CCND1,CCNE1,CCND1.CCNE1,-7.161001,-7.164089,-7.170788,-7.162667,-7.162876,-7.171031,-7.174502,...,-5.115351,-5.453035,-5.462034,-6.670703,-6.663899,-6.629628,-6.601424,-6.610152,-6.620162,-6.465161
1,CCNA2,CCND1,CCNA2.CCND1,-7.166567,-6.576004,-4.910347,-3.324249,-4.055324,-6.747916,-7.174107,...,-3.204796,-5.130190,-6.588219,-6.425308,-4.948783,-3.990528,-3.759030,-4.371402,-5.777962,-6.639350
2,CCNB1,CCND1,CCNB1.CCND1,-7.001215,-6.132371,-4.179894,-3.271571,-4.976478,-7.117143,-7.174709,...,-3.082754,-4.666952,-5.926837,-6.378951,-5.197193,-4.207916,-3.691921,-4.232875,-5.472432,-6.318435
3,CCNA2,CCNE1,CCNA2.CCNE1,-7.174693,-7.172061,-7.158068,-7.144946,-7.168169,-7.174597,-7.174724,...,-5.054388,-6.196282,-6.659328,-6.671752,-6.630302,-6.466193,-6.558847,-6.599185,-6.666088,-6.671980
4,CCNB1,CCNE1,CCNB1.CCNE1,-7.174528,-7.172478,-7.153281,-7.147242,-7.170332,-7.174685,-7.174724,...,-5.095836,-6.125032,-6.509610,-6.666062,-6.554793,-6.578068,-6.567707,-6.552521,-6.665822,-6.671197


In [10]:
pair_matrix.shape


(84942, 905)

In [16]:
with open(ARTIFACTS_DIR / 'controls' / 'positive_controls.pkl', 'rb') as file:
    pos = pickle.load(file)
list(pd.DataFrame(pos[0]).iloc[1])

['Cyclin–CDK complexes',
 'CCND1.CCNB1',
 np.float64(-7.0012150419808545),
 np.float64(-7.062737373036463),
 np.float64(-7.151570567308305),
 np.float64(-7.139621024341011),
 np.float64(-7.111579506344592),
 np.float64(-7.144891253661753),
 np.float64(-7.173987227949322),
 np.float64(-6.132371041404292),
 np.float64(-5.945426808702123),
 np.float64(-6.163761772074767),
 np.float64(-5.790490786786691),
 np.float64(-5.800849757805399),
 np.float64(-6.287784488161814),
 np.float64(-7.132976888359672),
 np.float64(-4.179893982957593),
 np.float64(-3.678010105180583),
 np.float64(-3.831497376806499),
 np.float64(-2.994781099569507),
 np.float64(-3.0470815895008307),
 np.float64(-4.378648116158997),
 np.float64(-6.564410985906272),
 np.float64(-3.2715709097070707),
 np.float64(-2.609747791508637),
 np.float64(-2.794479419212118),
 np.float64(-1.6192281437022313),
 np.float64(-1.5258836486028708),
 np.float64(-2.995952097308818),
 np.float64(-5.626058505156898),
 np.float64(-4.976478285484804